In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))


# ── shared/mlfp05/ex_2.py ──
"""
Shared utilities for MLFP05 Exercise 2 — CNNs on CIFAR-10.

Common infrastructure used by all technique files:
  - CIFAR-10 data loading and normalisation
  - Device detection and precision settings
  - ExperimentTracker / ModelRegistry setup
  - LightningModule wrapper for training
  - Visualisation helpers
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pytorch_lightning as pl
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.engines.model_registry import LocalFileArtifactStore
from kailash_ml.types import MetricSpec

# ── Environment & Reproducibility ────────────────────────────────────
setup_environment()
torch.manual_seed(42)
np.random.seed(42)
pl.seed_everything(42, workers=True)

DEVICE = get_device()
# Notebook-safe path resolution: __file__ is undefined when this module is
# inlined into a Colab notebook by scripts/generate_selfcontained_notebook.py.
try:
    _HERE = Path(__file__).resolve()
    REPO_ROOT = _HERE.parents[2]
    ARTIFACT_DIR = _HERE.parent
except NameError:
    REPO_ROOT = Path.cwd()
    ARTIFACT_DIR = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"

# The ModelRegistry stores artifact files (model.pkl, model.onnx) in an
# ArtifactStore. We construct the store explicitly so exercises can attach
# an ONNX serving artifact to a registered version (see attach_onnx_artifact).
ARTIFACT_STORE = LocalFileArtifactStore(".kailash_ml/artifacts")

N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

# ── CIFAR-10 class labels (used for visualisation and apply sections) ─
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Per-channel normalisation statistics (CIFAR-10 population)
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)


# ═══════════════════════════════════════════════════════════════════════
# Data Loading
# ═══════════════════════════════════════════════════════════════════════
def load_cifar10() -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    DataLoader,
    DataLoader,
]:
    """Load and normalise the full CIFAR-10 dataset.

    Returns:
        X_train, y_train, X_val, y_val, train_loader, val_loader
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_train = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_train = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))],
        dtype=torch.long,
    )
    X_val = torch.stack([val_set[i][0] for i in range(len(val_set))])
    y_val = torch.tensor(
        [val_set[i][1] for i in range(len(val_set))],
        dtype=torch.long,
    )

    # Per-channel normalisation
    X_train = (X_train - CIFAR_MEAN) / CIFAR_STD
    X_val = (X_val - CIFAR_MEAN) / CIFAR_STD

    train_ds = TensorDataset(X_train, y_train)
    val_ds = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    print(
        f"CIFAR-10: train {tuple(X_train.shape)}, val {tuple(X_val.shape)}, "
        f"classes={N_CLASSES}: {CLASS_NAMES}"
    )
    return X_train, y_train, X_val, y_val, train_loader, val_loader


# ═══════════════════════════════════════════════════════════════════════
# Kailash Engine Setup
# ═══════════════════════════════════════════════════════════════════════
async def setup_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Initialise ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry.

    Returns:
        conn, tracker, experiment_name, registry, has_registry
    """
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker and
    # ModelRegistry use incompatible _kml_model_versions schemas. Route them
    # to separate sqlite files until upstream fixes the conflict.
    db = f"sqlite:///{db_name}"
    registry_db_name = (
        db_name.replace(".db", "_registry.db")
        if db_name.endswith(".db")
        else db_name + "_registry.db"
    )
    registry_db = f"sqlite:///{registry_db_name}"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn, artifact_store=ARTIFACT_STORE)
    return conn, tracker, "m5_cnns", registry, True


def init_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Sync wrapper for setup_engines."""
    return asyncio.run(setup_engines(db_name))


# ═══════════════════════════════════════════════════════════════════════
# Lightning Training Infrastructure
# ═══════════════════════════════════════════════════════════════════════
class LitCNN(pl.LightningModule):
    """Lightning wrapper for any nn.Module classifier.

    Tracks per-epoch training loss and validation accuracy for later
    logging to ExperimentTracker and visualisation with ModelVisualizer.
    """

    def __init__(self, model: nn.Module, lr: float = 1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        self.train_losses: list[float] = []
        self.val_accs: list[float] = []
        self._batch_losses: list[float] = []
        self._val_correct = 0
        self._val_total = 0

    def training_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        loss = F.cross_entropy(logits, y)
        self._batch_losses.append(loss.item())
        return loss

    def on_train_epoch_end(self):
        if self._batch_losses:
            self.train_losses.append(float(np.mean(self._batch_losses)))
            self._batch_losses = []

    def validation_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        self._val_correct += int((logits.argmax(dim=-1) == y).sum().item())
        self._val_total += int(y.size(0))

    def on_validation_epoch_end(self):
        if self._val_total > 0:
            self.val_accs.append(self._val_correct / self._val_total)
            self._val_correct = 0
            self._val_total = 0

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.lr)


def get_precision_setting() -> str:
    """Return the optimal Lightning precision string for the current backend.

    Routes through ``kailash_ml.device()`` so MPS / CUDA / ROCm / Intel XPU /
    CPU are all selected by the same policy the rest of the platform uses.
    BackendInfo.precision is the canonical answer ("16-mixed" on Apple MPS
    + Tensor-Core GPUs, "32" on older GPUs and CPU).
    """
    import kailash_ml as km

    backend = km.device()
    print(f"  {backend.backend} detected -- using {backend.precision} precision")
    return backend.precision


def get_accelerator() -> str:
    """Return the Lightning accelerator string for the current backend."""
    import kailash_ml as km

    return km.device().accelerator


PRECISION = get_precision_setting()
ACCELERATOR = get_accelerator()


async def train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Train a CNN with Lightning and log metrics to ExperimentTracker.

    Uses the kailash-ml 1.1.1 ``tracker.track(...)`` async context manager.
    On normal exit the run is marked FINISHED; on exception it is marked FAILED.
    """
    lit = LitCNN(model, lr=lr)
    trainer = pl.Trainer(
        max_epochs=epochs,
        accelerator=ACCELERATOR,
        precision=PRECISION,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
    )

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "architecture": name,
                "lr": str(lr),
                "epochs": str(epochs),
                "batch_size": str(BATCH_SIZE),
                "dataset_size": str(len(train_loader.dataset)),
                "precision": PRECISION,
                "accelerator": ACCELERATOR,
            }
        )

        trainer.fit(lit, train_loader, val_loader)

        for epoch_idx, loss in enumerate(lit.train_losses):
            await run.log_metric("train_loss", loss, step=epoch_idx + 1)
        for epoch_idx, acc in enumerate(lit.val_accs):
            await run.log_metric("val_accuracy", acc, step=epoch_idx + 1)

        await run.log_metrics(
            {
                "final_train_loss": lit.train_losses[-1],
                "final_val_accuracy": lit.val_accs[-1],
            }
        )

    print(
        f"  [{name}] final train loss={lit.train_losses[-1]:.4f}  "
        f"val acc={lit.val_accs[-1]:.3f}"
    )
    return lit.train_losses, lit.val_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            lr,
            epochs,
        )
    )


# ═══════════════════════════════════════════════════════════════════════
# Model Registration
# ═══════════════════════════════════════════════════════════════════════
async def register_model_async(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Register a trained model with metrics in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="val_accuracy", value=final_acc),
            MetricSpec(name="epochs", value=float(epochs)),
            MetricSpec(name="batch_size", value=float(BATCH_SIZE)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={final_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Sync wrapper for register_model_async."""
    return asyncio.run(
        register_model_async(registry, name, model, final_loss, final_acc, epochs)
    )


# ═══════════════════════════════════════════════════════════════════════
# Visualisation Helpers
# ═══════════════════════════════════════════════════════════════════════
def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance.

    Delegates to the canonical factory in shared.mlfp05 — the P2
    experimental notice is acknowledged there.
    """

    return _canonical()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
    x_label: str = "Epoch",
    y_label: str = "Value",
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(
        metrics=metrics,
        x_label=x_label,
        y_label=y_label,
    )
    fig.write_html(str(output_path))
    print(f"  Saved plot: {output_path}")


def count_parameters(model: nn.Module) -> int:
    """Count total trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters())


def denormalise_cifar(img_tensor: torch.Tensor) -> torch.Tensor:
    """Reverse CIFAR-10 normalisation for display.

    Args:
        img_tensor: shape (C, H, W) or (B, C, H, W), normalised

    Returns:
        Tensor with pixel values clipped to [0, 1]
    """
    if img_tensor.dim() == 3:
        mean = CIFAR_MEAN.squeeze(0)
        std = CIFAR_STD.squeeze(0)
    else:
        mean = CIFAR_MEAN
        std = CIFAR_STD
    return (img_tensor * std + mean).clamp(0, 1)


# ═══════════════════════════════════════════════════════════════════════
# Serving Helpers (ONNX export + InferenceServer)
# ═══════════════════════════════════════════════════════════════════════
IMAGE_SHAPE = (3, 32, 32)
N_PIXELS = 3 * 32 * 32


class FlatImageAdapter(nn.Module):
    """Wrap an image classifier so it accepts flat pixel rows.

    InferenceServer's ONNX runtime turns each request record into ONE row of
    a 2-D float array, so the exported graph must take ``(batch, 3072)``.
    This adapter reshapes rows back to ``(batch, 3, 32, 32)`` before calling
    the wrapped CNN. ``predict(X)`` is the method ``OnnxBridge.validate``
    calls to get the native (PyTorch) outputs for comparison.
    """

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x.reshape(-1, *IMAGE_SHAPE))

    def predict(self, X) -> np.ndarray:
        device = next(self.model.parameters()).device
        with torch.no_grad():
            x = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=device)
            return self.forward(x).cpu().numpy()


def images_to_records(images: torch.Tensor) -> list[dict[str, float]]:
    """Turn a batch of images into InferenceServer request records.

    One record per image, one key per pixel. Keys are zero-padded so the
    record order is the pixel order the ONNX graph expects.
    """
    rows = images.reshape(len(images), -1).cpu().tolist()
    return [{f"px{j:04d}": float(v) for j, v in enumerate(row)} for row in rows]


def attach_onnx_artifact(name: str, version: int, onnx_path: Path) -> None:
    """Store an exported ONNX file as ``model.onnx`` for a registered version.

    ``ModelRegistry.register_model`` saves the training artifact as
    ``model.pkl``; ``InferenceServer.from_registry(..., runtime="onnx")``
    loads ``model.onnx`` from the same ArtifactStore.

    OnnxBridge writes large weights to a sibling ``<file>.onnx.data`` file
    (ONNX external data); storing only the ``.onnx`` bytes would serve a model
    without its weights, so it is stored as ONE self-contained protobuf.
    """
    import onnx

    model_proto = onnx.load(str(onnx_path))  # pulls in any .onnx.data weights
    asyncio.run(
        ARTIFACT_STORE.save(name, version, model_proto.SerializeToString(), "model.onnx")
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP05 Exercise 2.3 — Production Pipeline: ONNX Export + InferenceServer
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this file, you will be able to:
#   - Explain WHY production teams ship ONNX instead of PyTorch
#     (dependency weight, hardware portability, multi-language runtimes)
#     in terms a non-technical manager can understand
#   - Export a trained CNN to ONNX format using kailash-ml's OnnxBridge
#   - Validate ONNX output matches PyTorch output (numerical fidelity)
#   - Register the ONNX artifact in ModelRegistry and serve predictions
#     through kailash-ml's InferenceServer
#   - Benchmark latency and throughput for production sizing
#   - Apply this to deploying the best model at a Singapore e-commerce
#     platform — latency targets, throughput planning, cost per inference
#
# PREREQUISITES: M5/ex_2/01_simple_cnn.py and 02_resnet_se.py (trained
#   CNN models, ModelRegistry registration)
# ESTIMATED TIME: ~25 min
#
# PHASES:
#   1. THEORY  — Why ONNX, what it solves, deployment landscape
#   2. BUILD   — Train a model and export to ONNX via OnnxBridge
#   3. TRAIN   — (integrated with BUILD — we need a trained model to export)
#   4. VISUALISE — Latency distribution, PyTorch vs ONNX comparison
#   5. APPLY   — E-commerce deployment: latency, throughput, cost
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from kailash_ml import OnnxBridge
from kailash_ml import InferenceServer



## PHASE 1 — THEORY: Why You Cannot Deploy PyTorch Directly


PyTorch = workshop (great for building, too heavy for shipping)
  ONNX = standard engine format (any mechanic can install)

  Benefits:
    - 50 MB vs 5 GB dependency footprint
    - Hardware-agnostic (CPU, GPU, ARM, Apple Silicon)
    - Language-agnostic (Python, Go, Java, C++, Rust)
    - Free optimisation (graph fusion, constant folding)
    - 2-3x latency improvement typical

  Pipeline: Train -> Export -> Validate -> Register -> Serve -> Monitor


In [ ]:
# Imagine you've built a perfect engine in your workshop. It runs
# beautifully on the test bench. Now you need to install it in a car
# that's already driving at 100 km/h. You can't bring the whole workshop
# along — you need to ship JUST the engine, in a standard format that
# any mechanic can install.
#
# ONNX (Open Neural Network Exchange) is that standard format for ML.
#
# WHY NOT DEPLOY PYTORCH DIRECTLY?
#
# 1. DEPENDENCY WEIGHT:
#    PyTorch + CUDA + all dependencies = 2-5 GB installed.
#    ONNX Runtime = ~50 MB. In a containerised deployment (Kubernetes),
#    this is the difference between 30-second cold starts and 2-second
#    cold starts.
#
# 2. HARDWARE PORTABILITY:
#    PyTorch weights CAN be loaded on a CPU (map_location="cpu"), but the
#    serving host still needs the full PyTorch stack built for that
#    hardware. ONNX Runtime runs the same .onnx file on CPU, NVIDIA GPU,
#    AMD GPU, Apple Silicon, ARM — any hardware with an execution provider.
#
# 3. LANGUAGE BARRIER:
#    Your model was trained in Python. Production services might be in
#    Go, Java, C++, or Rust. ONNX Runtime has native bindings for all.
#    No Python GIL bottleneck in production.
#
# 4. OPTIMISATION:
#    ONNX Runtime applies graph-level optimisations: operator fusion,
#    constant folding, memory planning. A model that takes 15ms in
#    PyTorch often takes 5-8ms in ONNX Runtime — free speedup.
#
# THE DEPLOYMENT PIPELINE:
#   Train (PyTorch) -> Export (ONNX) -> Validate -> Register (ModelRegistry)
#   -> Serve (InferenceServer / ONNX Runtime) -> Monitor (DriftMonitor)
#
# InferenceServer serves ONE registered model version per server:
#   server = await InferenceServer.from_registry(name, registry=registry,
#                                                version=v, runtime="onnx")
#   await server.start()          # loads that version's model.onnx
#   out = await server.predict({"records": [...]})   # {"predictions": [...]}
# Each record becomes one row of a 2-D float array fed to ONNX Runtime,
# so an image model is exported behind a small adapter that accepts flat
# pixel rows (FlatImageAdapter in shared/mlfp05/ex_2.py).

print("=" * 70)
print("  PHASE 1 — THEORY: Why ONNX for Production Deployment")
print("=" * 70)
print(
)



## PHASE 2+3 — BUILD + TRAIN: Model for Export


Residual block for the export model.


Squeeze-and-Excitation block for the export model.


ResNet+SE model for ONNX export.


In [ ]:
# We train a ResNetSE model (same as 02_resnet_se.py) and export it.
# In production, you'd load the best model from ModelRegistry instead.

print("=" * 70)
print("  PHASE 2+3 — BUILD + TRAIN: Prepare Model for Export")
print("=" * 70)


class ResBlock(nn.Module):

    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + identity)


class SEBlock(nn.Module):

    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        s = F.adaptive_avg_pool2d(x, 1).view(b, c)
        w = self.fc(s).view(b, c, 1, 1)
        return x * w


class ResNetSE(nn.Module):

    def __init__(self, n_classes: int = N_CLASSES):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.block1 = ResBlock(32)
        self.se1 = SEBlock(32)
        self.block2 = ResBlock(32)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.block1(x)
        x = self.se1(x)
        x = self.block2(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


# Load data and train
X_train, y_train, X_val, y_val, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()

print(f"\nTraining ResNetSE for ONNX export ({EPOCHS} epochs)...")
resnet_se = ResNetSE()
resnet_losses, resnet_accs = train_model(
    resnet_se,
    "ResNetSE_for_export",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)



## DIAGNOSTIC CHECKPOINT — pre-export sign-off


Cross-entropy on one (images, labels) batch, on the model's device.


In [ ]:
# Running the instruments BEFORE export is deployment hygiene: you do
# not want to ship a model whose training was secretly pathological.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _ce_loss(m, batch):
    xb, yb = batch
    dev = next(m.parameters()).device
    return F.cross_entropy(m(xb.to(dev)), yb.to(dev))


print("\n── Pre-Export Diagnostic Report (ResNetSE) ──")
diag, findings = run_diagnostic_checkpoint(
    resnet_se,
    train_loader,
    _ce_loss,
    title="ResNetSE (pre-export)",
    train_losses=resnet_losses,
    show=False,
)
print_prescription_pad(findings, "ResNetSE (pre-export)")
export_blockers = [
    name
    for name, reading in findings.items()
    if isinstance(reading, dict) and reading.get("severity") == "CRITICAL"
]
print(f"  CRITICAL readings that would block export: {export_blockers or 'none'}")

# ══════ READING THE PRESCRIPTION PAD (key: see ex_1/01_standard_ae.py) ══════
# Treat this as a release GATE, not a learning-curve reading: a
# CRITICAL reading (e.g. exploding/vanishing gradients, a mostly-dead
# layer) should block export until explained; WARNINGs belong in the
# model card. The export itself is checked separately below — the ONNX
# graph must reproduce the PyTorch outputs (OnnxBridge.validate).



In [ ]:
# Register in ModelRegistry (model.pkl = the trained weights)
model_version = register_model(
    registry,
    "resnet_se_cifar10",
    resnet_se,
    resnet_losses[-1],
    resnet_accs[-1],
)



### Checkpoint 1: Model trained


In [ ]:
assert (
    resnet_accs[-1] > 0.4
), f"ResNetSE val accuracy {resnet_accs[-1]:.3f} too low for export"
print(f"\nModel ready: loss={resnet_losses[-1]:.4f}, val_acc={resnet_accs[-1]:.3f}")
print("--- Checkpoint 1 passed --- model trained and ready for export\n")



## ONNX Export via OnnxBridge


In [ ]:
print("=" * 70)
print("  ONNX EXPORT")
print("=" * 70)

bridge = OnnxBridge()
resnet_se.eval()
onnx_path = Path("ex_2_resnet_se.onnx")

# InferenceServer feeds ONNX Runtime one flat row per request record, so
# we export the CNN behind FlatImageAdapter: it takes (batch, 3072) pixel
# rows and reshapes them to (batch, 3, 32, 32). The adapter must be in
# eval mode — OnnxBridge restores the training flag it finds. The sample
# is a batch of TWO rows, not one: the exporter traces with torch.export,
# which treats size-1 dimensions as constants and can freeze the batch size.
# (The exporter may print an opset-conversion traceback and fall back to
# opset 18; that is log noise — export_result.success is the real signal.)
serving_model = FlatImageAdapter(resnet_se).eval()
sample = torch.randn(2, 3 * 32 * 32)
export_result = bridge.export(
    serving_model,
    "torch",
    output_path=onnx_path,
    sample_input=sample,
)
print(
    f"  OnnxBridge.export: success={export_result.success} "
    f"status={export_result.onnx_status} "
    f"time={export_result.export_time_seconds:.1f}s"
)
# torch 2.12's STRICT FX-decomposer (used by OnnxBridge) rejects a bare
# nn.ReLU() as a "mutated constant"; the LENIENT torch.onnx.export path
# (strict=False) handles it. Kailash-first means trying OnnxBridge first,
# then falling back honestly rather than shipping nothing.
if not export_result.success:
    print(
        "  OnnxBridge strict path failed (torch-version quirk: bare nn.ReLU). "
        "Falling back to torch.onnx.export (strict=False) for this hand-built model."
    )
    import onnx

    onnx_model = torch.onnx.export(
        serving_model,
        (sample,),
        str(onnx_path),
        input_names=["input"],
        output_names=["logits"],
        # dynamic batch: a static 2-row trace would freeze the batch dim and
        # break validation/serving with any other batch size
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=17,
        dynamo=False,  # legacy exporter (lenient); the strict FX path above rejects nn.ReLU
    )
    assert onnx_path.exists(), "fallback torch.onnx.export must write the file"
    export_result = type("R", (), {"success": True})()  # exported via fallback
assert export_result.success, f"OnnxBridge export failed: {export_result.error_message}"

# Attach the .onnx file to the version registered above, so the
# InferenceServer can load it from the registry by name + version.
attach_onnx_artifact("resnet_se_cifar10", model_version.version, onnx_path)
print(f"  Attached model.onnx to resnet_se_cifar10 v{model_version.version}")



### Checkpoint 2: ONNX file exists


In [ ]:
assert onnx_path.exists(), "ONNX file should exist after export"
onnx_size_kb = onnx_path.stat().st_size // 1024
print(f"  Wrote {onnx_path} ({onnx_size_kb} KB)")
print(
    "  The .onnx file contains the architecture AND learned weights.\n"
    "  Any ONNX Runtime can load and execute it without PyTorch."
)
print("\n--- Checkpoint 2 passed --- ONNX export complete\n")



## Numerical Validation: PyTorch vs ONNX Runtime


In [ ]:
print("=" * 70)
print("  NUMERICAL VALIDATION: PyTorch vs ONNX Runtime")
print("=" * 70)

# onnxruntime is required here: OnnxBridge.validate and InferenceServer
# both execute the .onnx graph with it.
import onnxruntime as ort

ort_available = True  # the benchmark sections below also report ONNX Runtime

# OnnxBridge.validate runs the native model (via its predict() method) and
# ONNX Runtime on the same rows and reports the largest output difference.
test_images = X_val[:100]
test_flat = test_images.reshape(len(test_images), -1).numpy().astype(np.float32)
validation = bridge.validate(serving_model, onnx_path, test_flat, tolerance=1e-3)
print(
    f"  OnnxBridge.validate: valid={validation.valid} "
    f"max_diff={validation.max_diff:.2e} mean_diff={validation.mean_diff:.2e} "
    f"(over {validation.n_samples} logits)"
)
assert validation.valid, f"ONNX output drifted from PyTorch: {validation.notes}"

if ort_available:
    ort_session = ort.InferenceSession(str(onnx_path))
    input_name = ort_session.get_inputs()[0].name

    # Do the two runtimes pick the same class?
    resnet_se.eval()
    with torch.no_grad():
        pt_logits = resnet_se(test_images).numpy()
    pt_preds = np.argmax(pt_logits, axis=-1)

    ort_logits = ort_session.run(None, {input_name: test_flat})[0]
    ort_preds = np.argmax(ort_logits, axis=-1)

    prediction_match = np.mean(pt_preds == ort_preds)
    print(
        f"  Prediction agreement: {prediction_match:.0%} ({int(prediction_match * 100)}/100)"
    )
    assert prediction_match >= 0.99, (
        f"PyTorch vs ONNX prediction mismatch: {prediction_match:.0%} agreement. "
        "Export may have lost fidelity."
    )
    print("  Numerical validation PASSED -- ONNX matches PyTorch")



## InferenceServer Setup


Load one registered ONNX model version and serve a batch request.


In [ ]:
print("\n" + "=" * 70)
print("  INFERENCE SERVER")
print("=" * 70)


indices = [0, 100, 500, 2000, 5000]
sample_images = X_val[indices]


async def serve_samples(name: str, version: int, images: torch.Tensor):
    server = await InferenceServer.from_registry(
        name, registry=registry, version=version, runtime="onnx"
    )
    await server.start()  # loads model.onnx from the registry's artifact store
    print(f"  InferenceServer status: {server.status}  ({name} v{version}, onnx)")
    response = await server.predict({"records": images_to_records(images)})
    await server.stop()
    return response


response = asyncio.run(
    serve_samples("resnet_se_cifar10", model_version.version, sample_images)
)
server_logits = np.asarray(response["predictions"])
server_preds = server_logits.argmax(axis=1)

# The same images through PyTorch directly, for comparison
resnet_se.eval()
with torch.no_grad():
    logits = resnet_se(sample_images)
    probs = F.softmax(logits, dim=-1)
    preds = logits.argmax(dim=-1)

print("\n  InferenceServer predictions (vs direct PyTorch):")
for i, idx in enumerate(indices):
    server_class = CLASS_NAMES[server_preds[i]]
    torch_class = CLASS_NAMES[preds[i].item()]
    true_class = CLASS_NAMES[y_val[idx].item()]
    confidence = probs[i][preds[i]].item()
    status = "CORRECT" if server_preds[i] == y_val[idx].item() else "WRONG"
    print(
        f"    Sample {idx}: server={server_class:>10s} torch={torch_class:>10s} "
        f"(conf={confidence:.2f}) | true={true_class:>10s} [{status}]"
    )



### Checkpoint 3: InferenceServer served real predictions


In [ ]:
assert server_logits.shape == (len(indices), N_CLASSES), server_logits.shape
assert np.array_equal(server_preds, preds.numpy()), (
    "InferenceServer (ONNX) and PyTorch disagree on the sample classes"
)
batch_acc_check = float(np.mean(server_preds == y_val[indices].numpy()))
print(f"\n  Served-sample accuracy: {batch_acc_check:.0%} ({len(indices)} images)")
print("--- Checkpoint 3 passed --- InferenceServer served the ONNX model\n")



## PHASE 4 — VISUALISE: Latency Benchmarks


In [ ]:
# Production ML is not just about accuracy — it is about SPEED and COST.
# A model that takes 500ms per prediction cannot serve a website that
# needs <100ms response time.

print("=" * 70)
print("  PHASE 4 — VISUALISE: Latency Benchmarks")
print("=" * 70)

N_BENCHMARK = 100
single_image = X_val[:1]
batch_16 = X_val[:16]

# PyTorch latency
resnet_se.eval()
pt_latencies_single = []
pt_latencies_batch = []

with torch.no_grad():
    # Warmup
    for _ in range(10):
        _ = resnet_se(single_image)

    for _ in range(N_BENCHMARK):
        t0 = time.perf_counter()
        _ = resnet_se(single_image)
        pt_latencies_single.append((time.perf_counter() - t0) * 1000)

    for _ in range(N_BENCHMARK):
        t0 = time.perf_counter()
        _ = resnet_se(batch_16)
        pt_latencies_batch.append((time.perf_counter() - t0) * 1000)

# ONNX Runtime latency
ort_latencies_single = []
ort_latencies_batch = []

if ort_available:
    single_np = single_image.reshape(1, -1).numpy().astype(np.float32)
    batch_np = batch_16.reshape(16, -1).numpy().astype(np.float32)

    # Warmup
    for _ in range(10):
        _ = ort_session.run(None, {input_name: single_np})

    for _ in range(N_BENCHMARK):
        t0 = time.perf_counter()
        _ = ort_session.run(None, {input_name: single_np})
        ort_latencies_single.append((time.perf_counter() - t0) * 1000)

    for _ in range(N_BENCHMARK):
        t0 = time.perf_counter()
        _ = ort_session.run(None, {input_name: batch_np})
        ort_latencies_batch.append((time.perf_counter() - t0) * 1000)

# Print benchmark results
pt_single_mean = np.mean(pt_latencies_single)
pt_single_p99 = np.percentile(pt_latencies_single, 99)
pt_batch_mean = np.mean(pt_latencies_batch)

print(f"\n  {'Metric':>30s} {'PyTorch':>12s}", end="")
if ort_available:
    print(f" {'ONNX RT':>12s} {'Speedup':>10s}")
else:
    print()
print("  " + "-" * 70)

metrics_to_print = [
    (
        "Single image (mean)",
        pt_single_mean,
        np.mean(ort_latencies_single) if ort_available else None,
    ),
    (
        "Single image (p99)",
        pt_single_p99,
        np.percentile(ort_latencies_single, 99) if ort_available else None,
    ),
    (
        "Batch of 16 (mean)",
        pt_batch_mean,
        np.mean(ort_latencies_batch) if ort_available else None,
    ),
    (
        "Per-image in batch",
        pt_batch_mean / 16,
        np.mean(ort_latencies_batch) / 16 if ort_available else None,
    ),
]

for label, pt_val, ort_val in metrics_to_print:
    line = f"  {label:>30s} {pt_val:>10.2f}ms"
    if ort_val is not None:
        speedup = pt_val / ort_val if ort_val > 0 else float("inf")
        line += f" {ort_val:>10.2f}ms {speedup:>9.1f}x"
    print(line)

# Throughput calculation
pt_throughput = 1000 / (pt_batch_mean / 16)  # images/second
print(f"\n  PyTorch throughput: {pt_throughput:,.0f} images/second")
if ort_available:
    ort_throughput = 1000 / (np.mean(ort_latencies_batch) / 16)
    print(f"  ONNX RT throughput: {ort_throughput:,.0f} images/second")

# Visualise latency distributions
fig_latency, axes = plt.subplots(1, 2, figsize=(14, 5))
fig_latency.suptitle("Inference Latency: PyTorch vs ONNX Runtime", fontsize=14)

# Single image latency distribution
axes[0].hist(
    pt_latencies_single, bins=30, alpha=0.7, label="PyTorch", color="steelblue"
)
if ort_available:
    axes[0].hist(
        ort_latencies_single, bins=30, alpha=0.7, label="ONNX RT", color="coral"
    )
axes[0].set_xlabel("Latency (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Single Image Inference")
axes[0].legend()
axes[0].axvline(
    np.mean(pt_latencies_single), color="steelblue", linestyle="--", alpha=0.5
)
if ort_available:
    axes[0].axvline(
        np.mean(ort_latencies_single), color="coral", linestyle="--", alpha=0.5
    )

# Batch latency distribution
axes[1].hist(pt_latencies_batch, bins=30, alpha=0.7, label="PyTorch", color="steelblue")
if ort_available:
    axes[1].hist(
        ort_latencies_batch, bins=30, alpha=0.7, label="ONNX RT", color="coral"
    )
axes[1].set_xlabel("Latency (ms)")
axes[1].set_ylabel("Count")
axes[1].set_title("Batch of 16 Inference")
axes[1].legend()

plt.tight_layout()
plt.savefig("ex_2_03_latency_benchmark.png", dpi=150, bbox_inches="tight")
plt.close(fig_latency)
print("\n  Saved: ex_2_03_latency_benchmark.png")

# Model size comparison
pytorch_size_mb = sum(p.numel() * p.element_size() for p in resnet_se.parameters()) / (
    1024 * 1024
)
print(f"\n  Model sizes:")
print(f"    PyTorch in-memory: {pytorch_size_mb:.2f} MB")
print(f"    ONNX file on disk: {onnx_size_kb / 1024:.2f} MB ({onnx_size_kb} KB)")



### Checkpoint 4: Benchmarks complete


In [ ]:
import os

assert os.path.exists("ex_2_03_latency_benchmark.png"), "Latency plot missing"
assert pt_single_mean > 0, "PyTorch latency should be positive"
print("\n--- Checkpoint 4 passed --- latency benchmarks complete\n")



## PHASE 5 — APPLY: E-Commerce Deployment Planning


DEPLOYMENT SIZING (based on benchmark data):

  Traffic Profile:
    Daily listings:          {DAILY_LISTINGS:>10,}
    Average per second:      {avg_per_second:>10.1f}
    Peak per second (3x):    {peak_per_second:>10.1f}
    Latency SLA:             {LATENCY_SLA_MS:>10} ms (page load)
    Model latency budget:    {MODEL_LATENCY_BUDGET_MS:>10} ms

  Model Performance ({runtime_name}):
    Single image latency:    {single_image_ms:>10.2f} ms
    Batched throughput:      {batched_per_sec:>10.0f} images/sec/replica
    Meets latency budget:    {"YES" if single_image_ms < MODEL_LATENCY_BUDGET_MS else "NO":>10s}

  OPTION A — CPU Deployment (ONNX Runtime, recommended for this model):
    Replicas needed:         {cpu_replicas:>10}
    Instance type:           {"CPU-2":>10s} (generic 2 vCPU, 4 GB instance)
    Monthly cost:            ${cpu_monthly:>9,.0f}
    Cost per inference:      ${cost_per_inference_cpu:>9.6f}
    Pros: Simple, no GPU driver headaches, easy horizontal scaling
    Cons: Higher latency, more replicas needed

  OPTION B — GPU Deployment (ONNX Runtime + CUDA):
    Replicas needed:         {replicas_for_peak:>10}
    Instance type:           {"GPU-T4":>10s} (generic T4-class GPU, 4 vCPU, 16 GB)
    Monthly cost:            ${gpu_monthly:>9,.0f}
    Cost per inference:      ${cost_per_inference_gpu:>9.6f}
    Pros: Lower latency, fewer replicas, room for larger models
    Cons: GPU driver management, less elastic scaling

  RECOMMENDATION FOR THE ENGINEERING MANAGER:
    This model ({count_parameters(resnet_se):,} params, {onnx_size_kb} KB ONNX) is
    small enough for CPU deployment. GPU is overkill unless you plan to
    upgrade to a larger model (ResNet-50, EfficientNet) later.

    Start with Option A (CPU):
      - Deploy ONNX file to {cpu_replicas} CPU replicas behind a load balancer
      - Set auto-scaling: scale up at 70% CPU, scale down at 30%
      - Monitor with kailash-ml DriftMonitor for accuracy degradation
      - Budget: ${cpu_monthly:,.0f}/month (~${cpu_monthly * 12:,.0f}/year)

    Compared to an ASSUMED manual-categorisation cost
    ($10,000/day = $300,000/month, illustrative):
      Savings: ${300000 - cpu_monthly:,.0f}/month = ${(300000 - cpu_monthly) * 12:,.0f}/year


In [ ]:
# SCENARIO: You are deploying the product categorisation CNN from
# 01_simple_cnn.py to production at a Singapore e-commerce platform.
# The engineering manager asks three questions:
#   1. "What hardware do we need?"
#   2. "How many images per second can we process?"
#   3. "What will it cost?"
#
# You need concrete answers based on your benchmark data.

print("=" * 70)
print("  PHASE 5 — APPLY: E-Commerce Deployment (Singapore Platform)")
print("=" * 70)

# Production parameters
DAILY_LISTINGS = 500_000
PEAK_MULTIPLIER = 3.0  # Peak traffic is 3x average
LATENCY_SLA_MS = 100  # Product page must load in <100ms, model is part of that
MODEL_LATENCY_BUDGET_MS = 30  # Model gets 30ms of the 100ms budget

# Calculate required throughput
avg_per_second = DAILY_LISTINGS / (24 * 3600)
peak_per_second = avg_per_second * PEAK_MULTIPLIER

# How many replicas needed?
if ort_available:
    single_image_ms = np.mean(ort_latencies_single)
    runtime_name = "ONNX Runtime"
    images_per_sec_per_replica = 1000 / single_image_ms
else:
    single_image_ms = pt_single_mean
    runtime_name = "PyTorch"
    images_per_sec_per_replica = 1000 / single_image_ms

# With batching (batch of 16), throughput is higher
if ort_available:
    batched_per_sec = 1000 / (np.mean(ort_latencies_batch) / 16)
else:
    batched_per_sec = pt_throughput

replicas_for_peak = int(np.ceil(peak_per_second / batched_per_sec))
replicas_for_peak = max(replicas_for_peak, 2)  # minimum 2 for redundancy

# Cost estimation (Singapore cloud pricing)
CPU_COST_PER_HOUR = 0.05  # illustrative on-demand rate, 2 vCPU instance
GPU_COST_PER_HOUR = 0.90  # illustrative on-demand rate, T4-class GPU instance

# CPU deployment (ONNX Runtime)
cpu_replicas = max(replicas_for_peak * 2, 4)  # CPU is slower, need more
cpu_monthly = cpu_replicas * CPU_COST_PER_HOUR * 24 * 30

# GPU deployment (PyTorch or ONNX Runtime with CUDA)
gpu_monthly = replicas_for_peak * GPU_COST_PER_HOUR * 24 * 30

cost_per_inference_cpu = cpu_monthly / (DAILY_LISTINGS * 30)
cost_per_inference_gpu = gpu_monthly / (DAILY_LISTINGS * 30)

print(
)



### Checkpoint 5: Apply section complete


In [ ]:
assert replicas_for_peak >= 1, "Should need at least 1 replica"
assert cost_per_inference_cpu > 0, "Cost per inference should be positive"
print("--- Checkpoint 5 passed --- deployment planning complete\n")



## Clean up


In [ ]:
await conn.close()



## REFLECTION


THEORY:
  [x] Why teams ship ONNX rather than PyTorch to production (2-5 GB
      footprint, hardware portability, Python-only runtime)
  [x] ONNX as the universal exchange format (50 MB, any hardware,
      any language, free graph optimisations)
  [x] The deployment pipeline: Train -> Export -> Validate -> Register
      -> Serve -> Monitor

  BUILD + TRAIN:
  [x] Exported ResNetSE to ONNX with OnnxBridge (behind a flat-input adapter)
  [x] Validated numerical fidelity with OnnxBridge.validate
  [x] Registered the ONNX artifact and served it with InferenceServer
  [x] ONNX file: {onnx_path} ({onnx_size_kb} KB)

  VISUALISE (the proof):
  [x] Latency distribution: PyTorch vs ONNX Runtime side-by-side
  [x] Single image: {pt_single_mean:.2f}ms (PyTorch){f" vs {np.mean(ort_latencies_single):.2f}ms (ONNX)" if ort_available else ""}
  [x] Throughput: {pt_throughput:,.0f} img/sec (PyTorch){f" vs {ort_throughput:,.0f} img/sec (ONNX)" if ort_available else ""}

  APPLY:
  [x] Singapore e-commerce deployment planning with concrete numbers
  [x] CPU vs GPU cost comparison for this model size
  [x] Recommendation: CPU deployment at ${cpu_monthly:,.0f}/month
  [x] Annual savings vs manual review: ${(300000 - cpu_monthly) * 12:,.0f}

  KEY INSIGHT: The model is just one artifact in the deployment pipeline.
  ONNX export, numerical validation, latency benchmarking, capacity
  planning, and cost analysis are what turn "it works on my laptop" into
  "it's running in production." An engineering manager does not care about
  your model's val_accuracy — they care about latency, throughput, cost,
  and reliability.

  Next: In 04_hyperparameter_study.py, you'll explore the accuracy-vs-cost
  tradeoff with systematic learning rate and augmentation experiments...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

